<small>Reconocimiento de Patrones · Licenciatura en Ciencias de la Computación · Universidad de Sonora<br>
Julio Waissman · <a href="https://ml-unison.github.io">ml-unison.github.io</a> · Licencia CC BY-SA 4.0</small>

---

# Laboratorio 6 · Diferenciación automática y entrenamiento con PyTorch

**Unidad 6**

PyTorch calcula el gradiente de cualquier pérdida con una sola línea, `perdida.backward()`. En este laboratorio vas a entender qué hace esa línea y a usarla con soltura. En la **parte A** verás cómo PyTorch anota cada operación en un **grafo de cómputo** y lo recorre al revés para obtener los gradientes (la **diferenciación automática**): obtendrás con `backward()` el gradiente de la regresión logística y lo compararás con la fórmula; escribirás un descenso por gradiente a mano, cuidando que los gradientes se **acumulan**; agregarás al grafo una operación nueva con su propia derivada; verificarás gradientes con **diferencias finitas** y verás por qué no se entrena así; y construirás un perceptrón multicapa con la **inicialización** de Xavier o de He según su activación, que entrenarás con descenso por gradiente estocástico escrito por ti, sin optimizador. En la **parte B** escribirás el ciclo de entrenamiento estándar, con `DataLoader` y `torch.optim`, y una red convolucional pequeña con una arquitectura dada.

**Cómo se trabaja.** Completa las celdas marcadas con `...`. Después de cada pregunta hay una celda `grader.check("p1")` que verifica tu respuesta: si algo falla, te dice qué. Las preguntas marcadas como *reflexión* se responden por escrito en la celda indicada.

**Reglas.** En la parte A los gradientes se obtienen siempre con `backward()` (o, en la pregunta 4, con diferencias finitas): las fórmulas escritas a mano sirven solo para comparar, y no se usa `torch.optim`. Las operaciones sobre los datos deben ser vectorizadas: los únicos ciclos permitidos son los de los pasos, épocas y minilotes del entrenamiento, el de las capas de la red y el de las entradas en la verificación por diferencias finitas. En la parte B escribe los ciclos de entrenamiento tú mismo, sin bibliotecas de alto nivel.

**Cómo se entrega.** Ejecuta la libreta de principio a fin (*Entorno de ejecución → Reiniciar y ejecutar todo*), revisa que no haya errores y descarga el archivo `.ipynb`. Toda la libreta corre en CPU en menos de un minuto: no hace falta GPU.

Las verificaciones de esta libreta son solo una parte de la revisión: al calificar se usan pruebas adicionales.

In [ ]:
import os, sys

RAMA = "main"  # rama del repositorio del curso de la que se toman paquete, datos y pruebas
os.environ["RDP_RAMA"] = RAMA
if "google.colab" in sys.modules:
    %pip install -q otter-grader "rdp @ git+https://github.com/ml-unison/ml-unison.github.io@{RAMA}#subdirectory=paquete"

In [ ]:
# Verificador de respuestas (otter-grader). Usa las pruebas públicas del laboratorio:
# la copia local si existe; si no (en Colab), las descarga del repositorio del curso.
import pathlib, urllib.request
import otter

PRUEBAS = pathlib.Path("tests") / "u06-lab06-backprop"
if not PRUEBAS.exists():
    PRUEBAS.mkdir(parents=True)
    for _p in ['p1.py', 'p2.py', 'p3.py', 'p4.py', 'p5.py', 'p6.py', 'p7.py', 'p8.py']:
        urllib.request.urlretrieve(
            f"https://raw.githubusercontent.com/ml-unison/ml-unison.github.io/{RAMA}/unidades/06-redes-neuronales/laboratorios/tests/u06-lab06-backprop/{_p}",
            PRUEBAS / _p)
grader = otter.Notebook(tests_dir=str(PRUEBAS))

In [ ]:
import copy
import time

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

import rdp

rdp.estilo.usar()
torch.manual_seed(2011)
dispositivo = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Los datos y el problema

Usaremos los **dígitos escritos a mano** que ya conoces de la Unidad 4 (*Optical Recognition of Handwritten Digits*, E. Alpaydin y C. Kaynak, repositorio UCI, 1998; viene incluido en scikit-learn como `load_digits`). Son 1,797 imágenes de 8×8 de los dígitos 0 a 9, escritos por 43 personas. Cada imagen original era un mapa de bits de 32×32; se dividió en bloques de 4×4 y en cada bloque se contó cuántos píxeles estaban encendidos, de modo que cada uno de los 64 píxeles es un entero entre 0 y 16. Lo dividimos entre 16 para que quede entre 0 y 1.

**El problema.** Reconocer el dígito a partir de los 64 píxeles: clasificación en 10 clases. Usamos la misma partición estratificada que en la Unidad 4 (25 % para prueba, `random_state=2011`). Como referencia, la regresión logística multinomial con `C=10` acierta en 0.971 de las imágenes de prueba. ¿Puede una red neuronal entrenada por nosotros hacerlo mejor?

In [ ]:
digitos = load_digits()
X_todo = digitos.data / 16.0
y_todo = digitos.target

X_ent, X_pru, y_ent, y_pru = train_test_split(X_todo, y_todo, test_size=0.25, stratify=y_todo, random_state=2011)
print(f"entrenamiento: {X_ent.shape} · prueba: {X_pru.shape}")

logistica = LogisticRegression(C=10, max_iter=5000).fit(X_ent, y_ent)
print(f"logística (C = 10), exactitud en prueba: {logistica.score(X_pru, y_pru):.3f}")

fig, ejes = plt.subplots(2, 10, figsize=(11, 3.2), layout="constrained")
for ax, imagen, etiqueta in zip(ejes.ravel(), X_ent[:20], y_ent[:20]):
    ax.imshow(imagen.reshape(8, 8), cmap="gray_r")
    ax.set_title(str(etiqueta))
    ax.axis("off")
fig.suptitle("Veinte imágenes de entrenamiento con su etiqueta");

# Parte A · Diferenciación automática con PyTorch

## El grafo de cómputo

Entrenar una red es minimizar la pérdida promedio

$$L(\theta) = \frac1M \sum_{i=1}^{M} \ell\bigl(y^{(i)}, \hat y^{(i)}\bigr)$$

por descenso por gradiente, donde $\theta$ son todos los parámetros (pesos y sesgos), $M$ el número de datos, $\hat y^{(i)}$ la predicción del modelo para el dato $i$ y $\ell$ la pérdida de un dato. Para dar un paso hace falta el gradiente $\nabla_\theta L$, con una entrada por cada parámetro: miles en este laboratorio, miles de millones en un modelo de lenguaje. Derivarlo a mano para cada arquitectura nueva sería lento y propenso a errores, y es justo lo que una biblioteca como PyTorch hace por nosotros.

La idea es la **diferenciación automática** (en PyTorch, el módulo `torch.autograd`, al que se le dice *autograd*). Cada operación que se hace con un tensor marcado con `requires_grad=True` queda anotada en un **grafo de cómputo**: los nodos son las operaciones (una suma, un producto, un seno) y las aristas son los tensores que pasan de una a otra. Cada nodo sabe derivar *su* operación, y nada más. Cuando se llama `L.backward()`, PyTorch recorre el grafo **de la salida hacia las entradas** aplicando la regla de la cadena nodo por nodo (por eso se llama diferenciación en **modo inverso**, o **retropropagación**) y deja en el atributo `.grad` de cada tensor de entrada la derivada $\partial L/\partial \theta$. El costo de esa pasada hacia atrás es del mismo orden que el de la pasada hacia adelante, **sin importar cuántos parámetros haya**: ese es el hecho que hace posible entrenar redes grandes.

Un ejemplo con dos números. Sea $f(x_1, x_2) = x_1 x_2 + \sin x_1$. A mano, $\partial f/\partial x_1 = x_2 + \cos x_1$ y $\partial f/\partial x_2 = x_1$; en $(x_1, x_2) = (2, 3)$ valen $3 + \cos 2 \approx 2.5839$ y $2$. El grafo tiene tres nodos: un producto $u = x_1 x_2$, un seno $v = \sin x_1$ y una suma $f = u + v$. Como $x_1$ se usa en dos nodos, su derivada es la **suma** de lo que llega por los dos caminos: $\partial f/\partial x_1 = x_2 \cdot 1 + \cos x_1 \cdot 1$.

In [ ]:
x1 = torch.tensor(2.0, requires_grad=True)
x2 = torch.tensor(3.0, requires_grad=True)
u = x1 * x2
v = torch.sin(x1)
f = u + v
print(f"f = {f.item():.4f}")
print("nodo que produjo f:", f.grad_fn)
print("nodos de los que viene:", f.grad_fn.next_functions)
f.backward()
print(f"∂f/∂x1 = {x1.grad.item():.4f}   (a mano: x2 + cos x1 = {3 + np.cos(2):.4f})")
print(f"∂f/∂x2 = {x2.grad.item():.4f}   (a mano: x1 = 2)")

El atributo `grad_fn` es el nodo del grafo que produjo cada tensor (`AddBackward0` para la suma, `MulBackward0` para el producto, `SinBackward0` para el seno). El grafo se construye **mientras** se ejecuta el código, operación por operación: si hay un `if` o un ciclo de Python, cada ejecución arma su propio grafo, el que corresponde a lo que realmente se calculó. Por eso se dice que el grafo de PyTorch es **dinámico**. Después de `backward()` el grafo se libera; para volver a derivar hay que volver a calcular.

## 1. El gradiente de la regresión logística

Empecemos con un modelo cuyo gradiente conocemos. Queremos distinguir los **ceros** de los demás dígitos con regresión logística: cada imagen es un vector $\mathbf x^{(i)} \in \mathbb R^{n}$ con $n = 64$ píxeles, $y^{(i)} = 1$ si la imagen es un cero y $y^{(i)} = 0$ si no, y el modelo, con pesos $\mathbf w \in \mathbb R^n$ y sesgo $b \in \mathbb R$, es

$$z^{(i)} = \mathbf w^\top \mathbf x^{(i)} + b, \qquad \hat y^{(i)} = \sigma\bigl(z^{(i)}\bigr), \qquad \sigma(z) = \frac{1}{1 + e^{-z}} .$$

La pérdida de un dato es la entropía cruzada binaria, $\ell = -\bigl[y \log \sigma(z) + (1 - y)\log\bigl(1 - \sigma(z)\bigr)\bigr]$. Usando que $-\log\sigma(z) = \log(1 + e^{-z}) = \log(1 + e^{z}) - z$ y que $-\log\bigl(1 - \sigma(z)\bigr) = \log(1 + e^{z})$, se simplifica a

$$\ell = \log\bigl(1 + e^{z}\bigr) - y\,z .$$

Esta segunda forma es la que se programa: la función $\log(1 + e^{z})$ se llama ***softplus*** y PyTorch la calcula de forma estable con `F.softplus`, sin desbordarse aunque $z$ valga miles. (La primera forma falla: con $z = -1000$, $\sigma(z)$ se redondea a 0 y $\log 0 = -\infty$.) También existe `F.binary_cross_entropy_with_logits(z, y)`, que calcula exactamente el promedio de esta $\ell$.

Derivando, $\partial \ell/\partial z = \sigma(z) - y$, y con la regla de la cadena se llega a la fórmula de clase: si $\mathbf X \in \mathbb R^{M \times n}$ es la matriz con un dato por renglón e $\mathbf y \in \mathbb R^M$ el vector de etiquetas,

$$\nabla_{\mathbf w} L = \frac1M\,\mathbf X^\top\bigl(\sigma(\mathbf X \mathbf w + b) - \mathbf y\bigr), \qquad \frac{\partial L}{\partial b} = \frac1M\sum_{i=1}^{M}\bigl(\sigma(z^{(i)}) - y^{(i)}\bigr),$$

donde $\sigma$ se aplica entrada por entrada. Vamos a pedirle a PyTorch ese gradiente sin darle la fórmula, y comprobar que llega al mismo resultado.

In [ ]:
X_t = torch.tensor(X_ent, dtype=torch.float64)                  # M × n, en doble precisión
es_cero = torch.tensor(y_ent == 0, dtype=torch.float64)         # 1 si la imagen es un cero
w0 = 0.1 * torch.randn(64, dtype=torch.float64, generator=torch.Generator().manual_seed(1))
b0 = torch.tensor(-0.5, dtype=torch.float64)
print(f"X_t: {tuple(X_t.shape)} · ceros: {int(es_cero.sum())} de {len(es_cero)}")

**Pregunta 1.** Escribe `gradiente_logistica(X, y, w, b)`, que recibe los tensores `X` ($M \times n$), `y` (longitud $M$, con ceros y unos), `w` (longitud $n$) y `b` (un tensor de dimensión 0) y devuelve la tupla `(dw, db)` con $\nabla_{\mathbf w} L$ y $\partial L/\partial b$ **calculados con autograd**: haz copias de `w` y `b` que registren el grafo (`w.detach().clone().requires_grad_(True)`), calcula $L$ con la forma estable, llama `backward()` y devuelve los `.grad` de las copias. No uses la fórmula de arriba (sirve solo para comprobar) y no modifiques `w` ni `b`.

In [ ]:
def gradiente_logistica(X, y, w, b):
    """Gradiente de la entropía cruzada binaria media respecto a w y b, obtenido con backward()."""
    ...


dw, db = gradiente_logistica(X_t, es_cero, w0, b0)
residuo = torch.sigmoid(X_t @ w0 + b0) - es_cero                 # σ(Xw + b) − y
print("¿coincide dw con Xᵀ(σ(Xw + b) − y)/M?", torch.allclose(dw, X_t.T @ residuo / len(es_cero)))
print("¿coincide db con el promedio de σ(z) − y?", torch.allclose(db, residuo.mean()))

In [ ]:
grader.check("p1")

## 2. Los gradientes se acumulan

`backward()` no **reemplaza** el contenido de `.grad`: le **suma** el gradiente nuevo. Míralo con $L(\boldsymbol\theta) = \theta_1^2 + \theta_2^2$, cuyo gradiente es $2\boldsymbol\theta$: si se deriva tres veces sin limpiar, el gradiente aparece multiplicado por 1, 2 y 3.

In [ ]:
theta = torch.tensor([1.0, -2.0], requires_grad=True)
for intento in range(1, 4):
    L = (theta ** 2).sum()
    L.backward()
    print(f"después del backward {intento}: theta.grad = {theta.grad.tolist()}   (∇L = {(2 * theta).tolist()})")

La acumulación es deliberada: si una pérdida es suma de partes que no caben juntas en memoria (por ejemplo, un minilote grande partido en pedazos), se puede llamar `backward()` en cada parte y al final `.grad` tiene el gradiente de la suma. Pero en el descenso por gradiente cada paso necesita **solo** el gradiente en el punto actual, así que antes de cada `backward()` hay que limpiar: `theta.grad = None` (o `theta.grad.zero_()`). Eso es lo que hace `optimizador.zero_grad()` en el ciclo de entrenamiento de la parte B.

Hay otro detalle: la actualización $\boldsymbol\theta \leftarrow \boldsymbol\theta - \eta\,\nabla L(\boldsymbol\theta)$, con $\eta$ la tasa de aprendizaje, **no** debe quedar en el grafo, porque no es parte del modelo. Por eso se escribe dentro de un bloque `with torch.no_grad():` y modificando el tensor en su lugar (`theta -= tasa * theta.grad`). Fuera de ese bloque PyTorch se niega a modificar en su lugar un tensor que registra el grafo; y si se escribe `theta = theta - tasa * theta.grad`, el nombre `theta` pasa a apuntar a un tensor **nuevo**, resultado de una operación, cuyo `.grad` ya no se llena. Por último, `t.detach()` devuelve los mismos valores que `t` pero sin historia en el grafo, y `t.item()` saca el número de un tensor de un solo elemento: así se guardan resultados sin arrastrar el grafo.

**Pregunta 2.** Escribe `descenso_gradiente(perdida, theta0, tasa=0.1, pasos=100)`, que minimiza una función `perdida` (recibe un tensor $\boldsymbol\theta$ y devuelve un tensor de un solo número) con `pasos` pasos de descenso por gradiente a partir de `theta0`, sacando cada gradiente con `backward()`. Devuelve la tupla `(theta, historia)`: el $\boldsymbol\theta$ final como un tensor **sin historia** (`requires_grad` falso) e `historia`, una lista de `float` con el valor de la pérdida **antes** de cada paso (de largo `pasos`). Trabaja sobre una copia: `theta0` no debe cambiar. No uses `torch.optim`.

In [ ]:
def descenso_gradiente(perdida, theta0, tasa=0.1, pasos=100):
    """Descenso por gradiente con autograd. Devuelve (theta, historia)."""
    ...


# Un tazón: L(θ) = ½‖θ − c‖², con mínimo en c = (3, −1)
c = torch.tensor([3.0, -1.0], dtype=torch.float64)
theta_final, historia_dg = descenso_gradiente(lambda t: 0.5 * ((t - c) ** 2).sum(), torch.zeros(2, dtype=torch.float64), tasa=0.1, pasos=50)
print("θ final:", theta_final.tolist(), "· pérdida inicial y final:", historia_dg[0], historia_dg[-1])

In [ ]:
grader.check("p2")

## 3. Una operación nueva para el grafo

Cada nodo del grafo hace dos cosas. **Hacia adelante** calcula su salida $s = \phi(x)$ a partir de su entrada $x$. **Hacia atrás** recibe $\partial L/\partial s$, la derivada de la pérdida respecto a su salida (que le llega de los nodos de más adelante), y devuelve $\partial L/\partial x = \partial L/\partial s \cdot \phi'(x)$, la regla de la cadena aplicada solo a su pedazo (entrada por entrada, si $x$ es un tensor). PyTorch ya trae los nodos de todas sus operaciones, pero se pueden agregar nuevos con una subclase de `torch.autograd.Function` que define dos métodos estáticos:

- `forward(ctx, x)` calcula la salida. Lo que se necesitará en la pasada hacia atrás se guarda con `ctx.save_for_backward(x)`; `ctx` es un objeto de contexto que PyTorch pasa a ambos métodos.
- `backward(ctx, grad_salida)` recibe `grad_salida` $= \partial L/\partial s$ (de la forma de la salida), recupera lo guardado con `x, = ctx.saved_tensors` y devuelve $\partial L/\partial x$ (de la forma de $x$).

La operación se usa con `Softplus.apply(x)`. Vamos a escribir así la *softplus* de la sección 1, $s(x) = \log(1 + e^x)$, cuya derivada es justamente la logística: $s'(x) = e^x/(1 + e^x) = \sigma(x)$. Para que no se desborde se usa la identidad

$$\log(1 + e^x) = \max(x, 0) + \log\bigl(1 + e^{-|x|}\bigr),$$

en la que la exponencial nunca recibe un número positivo. (Compruébala: si $x > 0$, $\log(1 + e^x) = \log\bigl(e^x(e^{-x} + 1)\bigr) = x + \log(1 + e^{-x})$.) La función `torch.log1p(t)` calcula $\log(1 + t)$ con precisión aun cuando $t$ es diminuto.

Para revisar un `backward` escrito a mano, PyTorch ofrece `torch.autograd.gradcheck(funcion, (x,))`, que compara el gradiente de tu `backward` con el que se obtiene por diferencias finitas (sección 4). Exige tensores de doble precisión (`torch.float64`), porque en precisión simple las diferencias finitas son demasiado imprecisas.

**Pregunta 3.** Completa `Softplus`: `forward` calcula $\log(1 + e^x)$ con la forma estable y guarda lo necesario con `ctx.save_for_backward`; `backward` devuelve `grad_salida` multiplicado por $\sigma(x)$ (puedes usar `torch.sigmoid`). Debe funcionar con tensores de cualquier forma, sin `inf` ni `nan` aunque las entradas valgan miles, y sin modificar su entrada.

In [ ]:
class Softplus(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x):
        ...

    @staticmethod
    def backward(ctx, grad_salida):
        ...


softplus = Softplus.apply

x_prueba = torch.linspace(-4, 4, 9, dtype=torch.float64, requires_grad=True)
print("¿coincide con F.softplus?", torch.allclose(softplus(x_prueba), F.softplus(x_prueba)))
print("gradcheck:", torch.autograd.gradcheck(softplus, (x_prueba,), raise_exception=False))

In [ ]:
grader.check("p3")

## 4. Diferencias finitas: para verificar, no para entrenar

Un error en un gradiente no produce ningún mensaje: el modelo simplemente aprende peor, o no aprende. Por eso un gradiente nuevo se **verifica** comparándolo con una aproximación numérica, como hace `gradcheck`. La **diferencia centrada**

$$\frac{\partial f}{\partial x_k} \approx \frac{f(\mathbf x + h\,\mathbf e_k) - f(\mathbf x - h\,\mathbf e_k)}{2h},$$

donde $\mathbf e_k$ es el vector con un 1 en la entrada $k$ y ceros en las demás y $h$ es un número pequeño, tiene error $O(h^2)$ (en el desarrollo de Taylor de los dos términos, los de orden $h$ se cancelan), mucho mejor que el $O(h)$ de la diferencia hacia adelante $\bigl(f(\mathbf x + h\,\mathbf e_k) - f(\mathbf x)\bigr)/h$.

Para comparar dos gradientes no sirve el error absoluto (no es lo mismo una diferencia de $10^{-6}$ entre gradientes de tamaño $10^{-8}$ que entre gradientes de tamaño $10^{3}$), así que se usa el **error relativo**

$$\operatorname{err}(\mathbf a, \mathbf b) = \frac{\lVert \mathbf a - \mathbf b\rVert}{\max\bigl(\lVert \mathbf a\rVert + \lVert \mathbf b\rVert,\ 10^{-12}\bigr)},$$

con la norma euclidiana de todas las entradas (abajo está `error_relativo`, ya escrita). En doble precisión, un gradiente correcto da errores relativos de $10^{-7}$ o menos; un error de programación suele dar $10^{-2}$ o más.

In [ ]:
def error_relativo(a, b):
    """‖a − b‖ / max(‖a‖ + ‖b‖, 1e-12), con la norma de todas las entradas."""
    a, b = torch.as_tensor(a, dtype=torch.float64), torch.as_tensor(b, dtype=torch.float64)
    return float(torch.linalg.norm(a - b) / max(float(torch.linalg.norm(a) + torch.linalg.norm(b)), 1e-12))

**Pregunta 4.** Escribe `gradiente_numerico(f, x, h=1e-5)`, que recibe una función `f` (toma un tensor de la forma de `x` y devuelve un número, como `float` o como tensor de un elemento) y un tensor `x` de doble precisión y de **cualquier forma**, y devuelve un tensor de la forma de `x` con la diferencia centrada en cada entrada. Aquí sí puedes recorrer las entradas con un ciclo: `x.numel()` es el número de entradas, y en un tensor contiguo `t.view(-1)[k]` es la entrada $k$ contando renglón por renglón. Al terminar, `x` debe quedar **exactamente** como estaba. Usa dos evaluaciones de `f` por entrada.

In [ ]:
def gradiente_numerico(f, x, h=1e-5):
    """Gradiente de f en x por diferencias centradas, entrada por entrada."""
    ...


gradiente_numerico(lambda v: (v ** 3).sum(), torch.tensor([1.0, 2.0, -1.0], dtype=torch.float64))

In [ ]:
grader.check("p4")

Comparemos los dos caminos en el gradiente de la pregunta 1 respecto a $\mathbf w$: autograd con una pasada hacia adelante y una hacia atrás, y tus diferencias centradas, que evalúan la pérdida dos veces por cada uno de los 64 pesos.

In [ ]:
def perdida_logistica(w):
    z = X_t @ w + b0
    return torch.mean(F.softplus(z) - es_cero * z)


inicio = time.perf_counter()
dw_auto, _ = gradiente_logistica(X_t, es_cero, w0, b0)
t_auto = time.perf_counter() - inicio
inicio = time.perf_counter()
dw_num = gradiente_numerico(perdida_logistica, w0)
t_num = time.perf_counter() - inicio
print(f"error relativo entre autograd y diferencias finitas: {error_relativo(dw_auto, dw_num):.1e}")
print(f"autograd: 1 pasada adelante + 1 atrás, {1000 * t_auto:.2f} ms")
print(f"diferencias finitas: {2 * w0.numel()} evaluaciones de la pérdida, {1000 * t_num:.2f} ms")

Las dos cosas coinciden, pero el costo no se parece. Si el modelo tiene $P$ parámetros, las diferencias centradas necesitan $2P$ evaluaciones de la pérdida **por cada paso** de entrenamiento, mientras que la retropropagación necesita una pasada hacia adelante y una hacia atrás, que cuesta aproximadamente lo mismo que dos o tres pasadas hacia adelante, **sea cual sea** $P$. La red de la sección 6 tiene 4,810 parámetros: serían 9,620 evaluaciones por paso en lugar de dos. Un modelo con mil millones de parámetros sería impensable. Además, las diferencias finitas son aproximadas (la reflexión 2 explora cuánto), mientras que autograd da el gradiente exacto salvo redondeo. Por eso las diferencias finitas se usan solo para **verificar**, en modelos pequeños y con pocos datos.

## 5. Un perceptrón multicapa como `nn.Module`

Un perceptrón multicapa con $L$ capas transforma un dato $\mathbf a^{[0]} = \mathbf x^{(i)}$ capa por capa:

$$\mathbf z^{[l]} = \mathbf W^{[l]}\mathbf a^{[l-1]} + \mathbf b^{[l]}, \qquad \mathbf a^{[l]} = g\bigl(\mathbf z^{[l]}\bigr), \qquad l = 1, \dots, L - 1,$$

y la última capa solo calcula $\mathbf z^{[L]} = \mathbf W^{[L]}\mathbf a^{[L-1]} + \mathbf b^{[L]}$. Aquí $n^{[l]}$ es el número de neuronas de la capa $l$ (con $n^{[0]} = n = 64$ píxeles y $n^{[L]} = K = 10$ clases), $\mathbf W^{[l]} \in \mathbb R^{n^{[l]} \times n^{[l-1]}}$ y $\mathbf b^{[l]} \in \mathbb R^{n^{[l]}}$ son los pesos y sesgos de la capa $l$, y $g$ es la **activación**, que se aplica entrada por entrada: la ReLU, $g(z) = \max(0, z)$, o la tangente hiperbólica, $g(z) = \tanh z$.

Conviene leer la red en dos partes. Las capas ocultas, de la 1 a la $L-1$, son una **ingeniería de características aprendida**: convierten los 64 píxeles en un vector $\mathbf a^{[L-1]}$ de atributos nuevos, y en lugar de diseñarlos a mano, se ajustan con los datos. La última capa es un **modelo lineal generalizado** sobre esos atributos: sus $K$ puntajes $\mathbf z^{[L]}$ (*logits*) pasan por la *softmax* para dar las probabilidades de las clases, exactamente como en la regresión logística multinomial. Por eso la pérdida es la misma, la entropía cruzada, y por eso una red sin capas ocultas ($L = 1$) **es** la regresión logística multinomial. En regresión, la última capa sería una regresión lineal con error cuadrático; para conteos, una regresión de Poisson.

En PyTorch, `nn.Linear(n_ent, n_sal)` guarda el peso en `.weight`, de forma $n_{\text{sal}} \times n_{\text{ent}}$ como $\mathbf W^{[l]}$, y el sesgo en `.bias`. Como en el código cada renglón es un dato, para un minilote $\mathbf A$ calcula $\mathbf A\,\mathbf W^\top + \mathbf b$, que es la fórmula de arriba aplicada a cada renglón.

### La inicialización: Xavier y He

Los pesos iniciales se sortean al azar, con media cero, pero la **varianza** importa mucho. Una neurona de la capa $l$ calcula $z^{[l]}_k = \sum_{j=1}^{n^{[l-1]}} w^{[l]}_{kj}\, a^{[l-1]}_j + b^{[l]}_k$. Si los pesos son independientes entre sí y de las activaciones, con media cero y varianza $\operatorname{Var}(w)$, y los sesgos valen cero, la varianza de esa suma de $n^{[l-1]}$ términos es

$$\operatorname{Var}\bigl(z^{[l]}_k\bigr) = n^{[l-1]}\operatorname{Var}(w)\;\mathbb E\bigl[(a^{[l-1]}_j)^2\bigr].$$

Si el factor $n^{[l-1]}\operatorname{Var}(w)$ es mayor que lo necesario, la escala de las señales se multiplica en cada capa y explota; si es menor, se encoge y se desvanece. Tras 10 o 20 capas cualquiera de los dos casos arruina el entrenamiento, y a los gradientes, que recorren las mismas capas al revés, les pasa lo mismo. La varianza correcta depende de la activación:

- **tanh (Xavier o Glorot).** Cerca de cero, $\tanh z \approx z$, así que $\mathbb E[a^2] \approx \operatorname{Var}(z^{[l-1]})$ y la escala se conserva si $n^{[l-1]}\operatorname{Var}(w) = 1$. En la pasada hacia atrás los gradientes cruzan la capa multiplicados por $\mathbf W^{[l]\top}$, y el mismo argumento pide $n^{[l]}\operatorname{Var}(w) = 1$. Glorot y Bengio (2010) propusieron el compromiso entre las dos condiciones: $\operatorname{Var}(w) = \dfrac{2}{n^{[l-1]} + n^{[l]}}$.
- **ReLU (He o Kaiming).** La ReLU anula la mitad negativa de sus entradas; si $z$ es simétrica alrededor de cero, $\mathbb E[a^2] = \tfrac12\operatorname{Var}(z^{[l-1]})$. Para compensar esa mitad perdida hay que duplicar la varianza: $\operatorname{Var}(w) = \dfrac{2}{n^{[l-1]}}$ (He y coautores, 2015).

Los sesgos empiezan en cero. En PyTorch: `nn.init.kaiming_normal_(capa.weight, nonlinearity="relu")`, `nn.init.xavier_normal_(capa.weight)` y `nn.init.zeros_(capa.bias)`; también puedes escribirlo tú, dentro de `torch.no_grad()`, con `torch.randn` multiplicado por la desviación estándar. Por omisión, `nn.Linear` usa otra regla: pesos uniformes de varianza $1/(3n^{[l-1]})$, seis veces menor que la de He.

**Pregunta 5.** Completa la clase `MLP`. `tamanos` es la lista $[n^{[0]}, n^{[1]}, \dots, n^{[L]}]$ y `activacion` es `"relu"` o `"tanh"`.

- En `__init__`, crea las $L$ capas `nn.Linear(tamanos[l - 1], tamanos[l])` y guárdalas en `self.capas`, un `nn.ModuleList` (para que PyTorch registre sus parámetros). Inicializa los pesos de **todas** las capas, incluida la última, con la regla de He si la activación es ReLU y con la de Xavier si es tanh (normal o uniforme, mientras la varianza sea la correcta), y los sesgos en cero.
- En `forward`, aplica las capas en orden con la activación `g` después de cada una **excepto la última**, y devuelve los puntajes $\mathbf z^{[L]}$ (sin *softmax*). La entrada es un tensor de $M \times n^{[0]}$.

In [ ]:
class MLP(nn.Module):
    def __init__(self, tamanos, activacion="relu"):
        super().__init__()
        self.activacion = activacion
        ...

    def forward(self, x):
        g = torch.relu if self.activacion == "relu" else torch.tanh
        ...


_ = torch.manual_seed(2011)
red = MLP([64, 32, 10])
print(red)
print("forma de la salida para 5 imágenes:", tuple(red(torch.zeros(5, 64)).shape))
ancha = MLP([1000, 500, 10])
print(f"desviación estándar de W[1]: {ancha.capas[0].weight.std().item():.4f}  (He: √(2/1000) = {np.sqrt(2 / 1000):.4f})")

In [ ]:
grader.check("p5")

Veamos la inicialización en acción. Abajo pasamos 500 vectores al azar por una red de 10 capas ocultas de 256 neuronas ReLU y medimos la desviación estándar de las activaciones $\mathbf a^{[l]}$ en cada capa, con tres inicializaciones: la de He (tu `MLP`), la que trae `nn.Linear` por omisión y pesos normales con desviación estándar fija de 0.01.

In [ ]:
def escalas_por_capa(red, x):
    """Desviación estándar de las activaciones a[l] de cada capa oculta."""
    escalas = []
    with torch.no_grad():
        for capa in red.capas[:-1]:
            x = torch.relu(capa(x))
            escalas.append(x.std().item())
    return escalas


_ = torch.manual_seed(2011)
entrada = torch.randn(500, 256)
profunda = MLP([256] * 11 + [10])
inicializaciones = {"He: Var(w) = 2/n[l−1]": escalas_por_capa(profunda, entrada)}
for capa in profunda.capas:
    capa.reset_parameters()                       # la regla por omisión de nn.Linear
inicializaciones["por omisión de nn.Linear"] = escalas_por_capa(profunda, entrada)
with torch.no_grad():
    for capa in profunda.capas:
        capa.weight.normal_(0.0, 0.01)
        capa.bias.zero_()
inicializaciones["desviación estándar fija de 0.01"] = escalas_por_capa(profunda, entrada)

fig, ax = plt.subplots(figsize=(8, 4.5))
for (nombre, escalas), color in zip(inicializaciones.items(), [rdp.estilo.AZUL, rdp.estilo.ROJO, rdp.estilo.GRIS]):
    ax.semilogy(range(1, 11), escalas, marker="o", color=color, label=nombre)
ax.set_xlabel("capa oculta l")
ax.set_ylabel("desviación estándar de a[l] (escala log)")
ax.set_title("La escala de las activaciones, capa por capa")
ax.legend();

Con la regla de He la escala se mantiene estable a lo largo de las diez capas. Con la regla por omisión se encoge en cada capa (hasta que los sesgos, que en esa regla no son cero, la sostienen), y con desviación estándar fija de 0.01 se desvanece casi por completo: la red produce prácticamente la misma salida para cualquier entrada y sus gradientes son diminutos. Con solo dos o tres capas, como en este laboratorio, el efecto es mucho menor; en redes profundas decide si la red entrena o no.

## 6. Entrenar a mano, sin optimizador

Con autograd y tu `MLP` ya tenemos todo para entrenar. El **descenso por gradiente estocástico** (SGD) usa en cada paso un **minilote** $\mathcal B$ de ejemplos y su pérdida promedio

$$L_{\mathcal B}(\theta) = \frac{1}{|\mathcal B|}\sum_{i \in \mathcal B} \ell\bigl(y^{(i)}, \hat y^{(i)}\bigr), \qquad \theta \leftarrow \theta - \eta\,\nabla_\theta L_{\mathcal B}(\theta),$$

donde $|\mathcal B|$ es el tamaño del minilote. En cada **época** se revuelven los $M$ ejemplos de entrenamiento y se recorren en minilotes de `tam_lote` ejemplos consecutivos de esa permutación (el último puede ser más chico); por cada minilote se da un paso. La pérdida $\ell$ es la entropía cruzada, que `F.cross_entropy(puntajes, etiquetas)` calcula a partir de los puntajes y las etiquetas enteras, aplicando la *softmax* por dentro de forma estable, y promediando sobre el minilote.

Para la actualización, `modelo.parameters()` recorre todos los tensores de parámetros del modelo (pesos y sesgos de todas las capas), y `modelo.zero_grad()` limpia los `.grad` de todos ellos.

In [ ]:
Xt_ent = torch.tensor(X_ent, dtype=torch.float32)
yt_ent = torch.tensor(y_ent, dtype=torch.long)
Xt_pru = torch.tensor(X_pru, dtype=torch.float32)
yt_pru = torch.tensor(y_pru, dtype=torch.long)


def exactitud(modelo, X, y):
    """Fracción de los datos que el modelo clasifica bien."""
    with torch.no_grad():
        return (modelo(X).argmax(dim=1) == y).float().mean().item()

**Pregunta 6.** Escribe `entrenar_sgd(modelo, X, y, tasa=0.1, epocas=30, tam_lote=32, semilla=2011)`, que entrena el `modelo` (modificándolo en su lugar) con descenso por gradiente estocástico y la entropía cruzada, y devuelve `historia`: una lista de `float` con la pérdida en **todo** el conjunto `X`, `y` al final de cada época, calculada dentro de `torch.no_grad()`. Revuelve con `generador = torch.Generator().manual_seed(semilla)`, creado una sola vez, y `torch.randperm(M, generator=generador)` al inicio de cada época. En cada paso: limpia los gradientes, calcula la pérdida del minilote, `backward()` y actualiza cada parámetro con `p -= tasa * p.grad` dentro de `torch.no_grad()`. **No uses `torch.optim`.**

In [ ]:
def entrenar_sgd(modelo, X, y, tasa=0.1, epocas=30, tam_lote=32, semilla=2011):
    """Entrena el modelo con SGD por minilotes, escrito a mano. Devuelve la historia de la pérdida."""
    ...


_ = torch.manual_seed(2011)
mlp_a = MLP([64, 64, 10])
inicio = time.perf_counter()
historia = entrenar_sgd(mlp_a, Xt_ent, yt_ent, tasa=0.1, epocas=30, tam_lote=32)
exactitud_a = exactitud(mlp_a, Xt_pru, yt_pru)
print(f"tiempo: {time.perf_counter() - inicio:.2f} s · exactitud en prueba: {exactitud_a:.3f}")

In [ ]:
grader.check("p6")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.semilogy(range(1, len(historia) + 1), historia, color=rdp.estilo.AZUL, marker="o")
ax.set_title("Entrenamiento del perceptrón con SGD escrito a mano")
ax.set_xlabel("época")
ax.set_ylabel("entropía cruzada en entrenamiento (escala log)");

Una red de 64 neuronas ocultas, entrenada con un ciclo de unas diez líneas, ya iguala o supera a la logística, y no escribimos ni una derivada: autograd las obtuvo del grafo. Los saltos ocasionales de la curva son el precio del descenso **estocástico**: algunos minilotes empujan los pesos en una dirección mala para el conjunto completo.

<!-- BEGIN QUESTION -->

**Reflexión 1 · Simetría.** ¿Qué pasa si se inicializan **todos** los pesos y sesgos en cero? Pruébalo: crea `MLP([64, 64, 10])`, pon todos sus parámetros en cero (`nn.init.zeros_(p)` para cada `p` de `modelo.parameters()`), entrénala con `entrenar_sgd` y mide la exactitud en prueba. Calcula la pérdida en un minilote, llama `backward()` y revisa los `.grad` de cada capa: ¿cuáles son cero y por qué? (Sigue la regla de la cadena: ¿de qué depende $\partial L/\partial \mathbf W^{[1]}$?) Luego prueba con todos los pesos iguales a una constante pequeña distinta de cero (por ejemplo 0.01): ¿las 64 neuronas ocultas aprenden cosas distintas? Demuestra, por inducción sobre los pasos de SGD, que si dos neuronas ocultas empiezan con los mismos pesos de entrada y de salida, siguen iguales para siempre. ¿Por qué la inicialización aleatoria rompe ese problema, y por qué la escala $2/n^{[l-1]}$ importa además del azar?

_Escribe aquí tu respuesta._

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

**Reflexión 2 · El precio de las diferencias finitas.** (a) ¿Cuántas evaluaciones de la pérdida necesitaría `gradiente_numerico` para **un** paso de SGD con el perceptrón de la pregunta 6? Mide cuánto tarda una evaluación de la pérdida en un minilote de 32 ejemplos y cuánto tardan una pasada hacia adelante más un `backward()`, y estima cuánto duraría una época con cada método. (b) Con la pérdida logística de la sección 4 (`perdida_logistica`), calcula el error relativo entre el gradiente de autograd y el numérico para $h = 10^{-1}, 10^{-2}, \dots, 10^{-12}$ y grafícalo en escala log-log. Explica la forma de la curva: ¿por qué el error baja al principio como $h^2$, y por qué vuelve a **subir** cuando $h$ es muy chico? (Pista: ¿cuántas cifras significativas tiene un `float64` y qué pasa al restar dos números casi iguales?) Estima el $h$ óptimo con un argumento de balance entre los dos errores. (c) La ReLU no es derivable en 0; ¿cuándo podría eso hacer fallar una verificación con diferencias finitas aunque el código esté bien?

_Escribe aquí tu respuesta._

<!-- END QUESTION -->

# Parte B · El ciclo de entrenamiento estándar

En la parte A revolviste los datos, partiste los minilotes y actualizaste cada parámetro con tus propias manos. PyTorch trae dos piezas que hacen ese trabajo. Un `DataLoader` entrega los datos en minilotes revueltos. Un **optimizador** de `torch.optim` guarda la lista de parámetros y hace la actualización: con `torch.optim.SGD` es exactamente tu `p -= tasa * p.grad`, y con métodos como el momento o Adam, que recuerdan información de los pasos anteriores, también guarda ese estado. El ciclo sigue siendo tuyo: una época recorre el cargador y, en cada minilote,

1. pone en cero los gradientes acumulados (`optimizador.zero_grad()`), porque, como viste en la pregunta 2, PyTorch los **suma** en cada `backward`;
2. calcula las predicciones y la pérdida;
3. llama `perdida.backward()` para obtener los gradientes;
4. llama `optimizador.step()` para actualizar los parámetros.

Antes del ciclo se llama `modelo.train()`, que activa el comportamiento de entrenamiento de capas como *dropout* o la normalización por lotes; para evaluar se usa `modelo.eval()` y `torch.no_grad()` (abajo está `evaluar`, ya escrita).

In [ ]:
def tensores(X, y):
    """Convierte las imágenes (M × 64) en un tensor de M × 1 × 8 × 8 y las etiquetas en enteros."""
    return torch.tensor(X, dtype=torch.float32).reshape(-1, 1, 8, 8), torch.tensor(y, dtype=torch.long)


datos_ent = TensorDataset(*tensores(X_ent, y_ent))
datos_pru = TensorDataset(*tensores(X_pru, y_pru))
cargador_pru = DataLoader(datos_pru, batch_size=256)


def evaluar(modelo, cargador):
    """Exactitud del modelo en los datos del cargador."""
    modelo.eval()
    aciertos, total = 0, 0
    with torch.no_grad():
        for xb, yb in cargador:
            xb, yb = xb.to(dispositivo), yb.to(dispositivo)
            aciertos += (modelo(xb).argmax(dim=1) == yb).sum().item()
            total += len(yb)
    return aciertos / total

**Pregunta 7.** Escribe `entrenar_epoca(modelo, cargador, optimizador, perdida)`, que pone el modelo en modo de entrenamiento, recorre una vez el `cargador` dando un paso del `optimizador` por minilote (mueve cada minilote a `dispositivo`) y devuelve la **pérdida media por ejemplo** de la época como `float`. `perdida` es una función como `F.cross_entropy` que devuelve el promedio del minilote; como el último minilote puede ser más chico, pondera cada promedio por el tamaño de su minilote: $\sum_{\mathcal B} |\mathcal B|\,L_{\mathcal B} \big/ \sum_{\mathcal B} |\mathcal B|$, donde la suma recorre los minilotes $\mathcal B$ de la época y $L_{\mathcal B}$ es la pérdida promedio del minilote. Usa `.item()` para sacar el número de un tensor.

In [ ]:
def entrenar_epoca(modelo, cargador, optimizador, perdida):
    """Una época de entrenamiento. Devuelve la pérdida media por ejemplo."""
    ...


# El mismo perceptrón de la parte A (64 → 64 → 10), ahora con nn.Sequential, la inicialización por omisión,
# un DataLoader y torch.optim.SGD
_ = torch.manual_seed(2011)
mlp = nn.Sequential(nn.Flatten(), nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 10)).to(dispositivo)
optimizador = torch.optim.SGD(mlp.parameters(), lr=0.1)
cargador_ent = DataLoader(datos_ent, batch_size=32, shuffle=True, generator=torch.Generator().manual_seed(2011))
for epoca in range(30):
    perdida_mlp = entrenar_epoca(mlp, cargador_ent, optimizador, F.cross_entropy)
exactitud_mlp = evaluar(mlp, cargador_pru)
print(f"MLP en PyTorch · pérdida de la última época: {perdida_mlp:.4f} · exactitud en prueba: {exactitud_mlp:.3f}")

In [ ]:
grader.check("p7")

## 8. Una red convolucional pequeña

El perceptrón trata los 64 píxeles como una lista sin orden: si revolviéramos los píxeles de todas las imágenes de la misma manera, aprendería exactamente igual. Una red **convolucional** aprovecha que los píxeles vecinos están relacionados: cada filtro de 3×3 recorre la imagen buscando el mismo patrón local en todas partes (pesos **compartidos**), y el *pooling* reduce la resolución a la mitad. Construye esta arquitectura, para entradas de $1 \times 8 \times 8$:

| capa | salida |
|---|---|
| convolución de 1 a 8 canales, filtro 3×3, `padding=1` | 8 × 8 × 8 |
| ReLU y *max pooling* de 2×2 | 8 × 4 × 4 |
| convolución de 8 a 16 canales, filtro 3×3, `padding=1` | 16 × 4 × 4 |
| ReLU y *max pooling* de 2×2 | 16 × 2 × 2 |
| aplanar | 64 |
| capa lineal de 64 a 10 | 10 puntajes |

La salida son los **puntajes** (sin *softmax*: `F.cross_entropy` la aplica por dentro, de forma estable).

**Pregunta 8.** Completa la clase `RedConvolucional` (las capas con parámetros se crean en `__init__` y se usan en `forward`; para ReLU y *pooling* puedes usar `F.relu` y `F.max_pool2d`). Después, en la celda que sigue, calcula a mano el número de parámetros de la red y escríbelo en `num_parametros` como un entero (no lo calcules con código: la celda de abajo lo compara con el conteo de PyTorch). Recuerda que una convolución de $c_{\text{ent}}$ a $c_{\text{sal}}$ canales con filtros de $3 \times 3$ tiene $c_{\text{sal}}(9\,c_{\text{ent}} + 1)$ parámetros.

In [ ]:
class RedConvolucional(nn.Module):
    def __init__(self):
        super().__init__()
        ...

    def forward(self, x):
        ...


num_parametros = ...

_ = torch.manual_seed(2011)
cnn = RedConvolucional().to(dispositivo)
print(f"parámetros (PyTorch): {sum(p.numel() for p in cnn.parameters()):,} · tu cuenta: {num_parametros}")
print("forma de la salida para 5 imágenes:", tuple(cnn(torch.zeros(5, 1, 8, 8, device=dispositivo)).shape))

In [ ]:
grader.check("p8")

Ahora la entrenamos con **Adam** y tu `entrenar_epoca`, 20 épocas.

In [ ]:
_ = torch.manual_seed(2011)
cnn = RedConvolucional().to(dispositivo)
optimizador = torch.optim.Adam(cnn.parameters(), lr=0.01)
cargador_ent = DataLoader(datos_ent, batch_size=32, shuffle=True, generator=torch.Generator().manual_seed(2011))
inicio = time.perf_counter()
for epoca in range(20):
    perdida_cnn = entrenar_epoca(cnn, cargador_ent, optimizador, F.cross_entropy)
exactitud_cnn = evaluar(cnn, cargador_pru)
print(f"tiempo: {time.perf_counter() - inicio:.1f} s · pérdida de la última época: {perdida_cnn:.4f}")

print(f"\n{'modelo':<34}{'parámetros':>11}{'exactitud':>11}")
print(f"{'logística (U4, C = 10)':<34}{650:>11,}{logistica.score(X_pru, y_pru):>11.3f}")
print(f"{'MLP, SGD a mano (parte A)':<34}{sum(p.numel() for p in mlp_a.parameters()):>11,}{exactitud_a:>11.3f}")
print(f"{'MLP, torch.optim.SGD (parte B)':<34}{sum(p.numel() for p in mlp.parameters()):>11,}{exactitud_mlp:>11.3f}")
print(f"{'red convolucional':<34}{sum(p.numel() for p in cnn.parameters()):>11,}{exactitud_cnn:>11.3f}")

La red convolucional usa menos de la mitad de los parámetros del perceptrón y acierta igual o más. (Los dos perceptrones no dan exactamente lo mismo aunque el algoritmo sea igual: el de la parte B usa la inicialización por omisión de `nn.Linear`, uniforme y de varianza menor que la de He, y el `DataLoader` revuelve los datos en otro orden.)

Abajo están los ocho filtros de la primera capa. Cada uno responde a un patrón local de 3×3, aprendido de los datos y no diseñado a mano; varios se parecen a detectores de bordes en alguna dirección. Con imágenes de 8×8 y tan pocos datos no hay que sobreinterpretarlos.

In [ ]:
filtros = [p for p in cnn.parameters() if p.dim() == 4][0].detach().cpu().numpy()[:, 0]  # pesos de la primera convolución
escala_f = np.abs(filtros).max()
fig, ejes = plt.subplots(1, 8, figsize=(11, 2.2), layout="constrained")
for i, (ax, filtro) in enumerate(zip(ejes, filtros)):
    ax.imshow(filtro, cmap="RdBu_r", vmin=-escala_f, vmax=escala_f)
    ax.set_title(f"filtro {i + 1}")
    ax.axis("off")
fig.suptitle("Filtros de 3×3 de la primera capa convolucional (rojo: peso positivo; azul: negativo)");

<!-- BEGIN QUESTION -->

**Reflexión 3.** Revuelve los 64 píxeles de **todas** las imágenes con una misma permutación fija (`perm = np.random.default_rng(0).permutation(64)`; usa `X_ent[:, perm]` y `X_pru[:, perm]`) y vuelve a entrenar tu perceptrón de la parte A (con `entrenar_sgd`) y la red convolucional con los mismos hiperparámetros. ¿Qué le pasa a la exactitud de cada una? Explica el resultado en términos de lo que cada arquitectura supone sobre los datos (su sesgo inductivo).

_Escribe aquí tu respuesta._

<!-- END QUESTION -->



---

To double-check your work, the cell below will rerun all of the autograder tests.

In [ ]:
grader.check_all()